# 10v2 — Host Holdout Mode-A: Corrected Reference Selection

Patches a real bug in `10_host_holdout_mode_a.ipynb` (v1, preserved unmodified): its
performance fix pre-capped each candidate's reference list to the first 30 entries **in raw
file order** before applying protocol filters (self/same-source/near-duplicate exclusion) --
"eligible references after row 30 are never considered", and file order is not a meaningful
ranking. This notebook fixes the ORDER of operations (filter fully on the complete reference
list first, rank what survives deterministically, THEN take the top `MAX_REFS_PER_CANDIDATE`)
while keeping the same tractability fix in spirit -- filtering itself is cheap (dict lookups over
the full list, not the previous version's O(n) triple-nested re-scan per protocol), so no
pre-cap is needed before it.

**Audit of the old bug (computed against the real v1 data before writing a single line of the
fix, `outputs/reports` below)**: of 126,573 (query, candidate) rows, 5,474 (4.3%) had a full
reference list longer than the old 30-item cap -- including 1,117 / 1,184 (94%) of TRUE
candidates (`enveda-np-examples` compounds turn out to be well-known natural products with
median 189 existing reference spectra each, not rare structures). **But zero rows (0.00%) had
their ELIGIBLE reference count drop below `MAX_REFS=5` because of the cap** -- so the old bug
never caused a coverage failure, only a potentially-suboptimal choice of WHICH 5 references got
used (arbitrary file order instead of adduct/CE-compatibility ranked). The freeze condition
"TRUE-CANDIDATE REFERENCE LOSS FROM CAP BUG = 0" was already satisfied; this notebook exists to
fix reference SELECTION QUALITY, not a coverage hole.

In [1]:
import sys, json, time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import lightgbm as lgb

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact, artifact_fingerprint, save_artifact, artifact_is_valid
from casmi.io.parquet import load_rows_by_offset
from casmi.candidates.cache import cached_mass_index
from casmi.candidates.mass_index import MassIndex
from casmi.candidates.generator import CandidateGenerator, dedupe_pool_to_connectivity
from casmi.spectra.preprocessing import remove_invalid_peaks, truncate_top_peaks
from casmi.spectra.library import build_reference_index
from casmi.spectra.deduplication import find_near_duplicate_references
from casmi.ranking.features import compute_pair_spectral_scores
from casmi.ranking.feature_policy import SPECTRAL_RANKING_FEATURE_POLICY, assert_no_prohibited_features
from casmi.ranking.evaluation import rank_candidates, ranking_metrics, true_candidate_rank, per_query_reciprocal_rank
from casmi.validation.checks import CheckResult, summarize_checks
from casmi.validation.baseline_registry import load_baseline, load_promoted_baseline, register_baseline
from casmi.validation.bootstrap import paired_bootstrap_report

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
N_BOOTSTRAP = 5000
HOST_INGEST_LIB = "enveda-np-examples"
MAX_REFS_PER_CANDIDATE = 5
BIN_WIDTH_DA = 0.1
PEAK_TOL_DA = 0.02
MAX_PEAKS_PER_SPECTRUM = 100
DEDUP_COSINE_THRESHOLD = 0.95
DEDUP_PRECURSOR_DIFF_DA = 0.01
LGBM_PARAMS = dict(
    objective="lambdarank", metric="ndcg", n_estimators=300, learning_rate=0.05,
    num_leaves=31, min_child_samples=20, random_state=SEED, verbosity=-1,
)

paths = get_project_paths()
CG_INTERIM = paths.interim / "candidate_generation"
CG_PROCESSED = paths.processed / "candidate_generation"
RANK_INTERIM = paths.interim / "spectral_ranking"
HOST_INTERIM = paths.interim / "host_holdout"
HOST_PROCESSED = paths.processed / "host_holdout"
BASELINE_REGISTRY_PATH = paths.processed / "baseline_registry.json"
REPORTS_DIR = paths.reports

MODEL_FEATURES = SPECTRAL_RANKING_FEATURE_POLICY.model_features
checks = []
print(f"MODEL_FEATURES: {MODEL_FEATURES}")

MODEL_FEATURES: ['abs_mass_error_ppm', 'cosine_max', 'cosine_top3_mean', 'modified_cosine_max', 'modified_cosine_top3_mean', 'peak_overlap_frac_max', 'peak_overlap_frac_top3_mean', 'neutral_loss_cosine_max', 'neutral_loss_cosine_top3_mean']


## 1. Reload host holdout artifacts (unaffected by the bug -- built from mass index only)

In [2]:
host_holdout_spectra = pd.read_parquet(HOST_PROCESSED / "host_holdout_spectra.parquet")
all_host_query_ids = host_holdout_spectra["query_id"].tolist()

train_meta = load_artifact("train_spectrum_metadata")
connectivity_folds = load_artifact("connectivity_folds")
with open(CG_PROCESSED / "candidate_generation_contract.json") as f:
    contract = json.load(f)
SELECTED_PPM = contract["molecule_policy"]["tolerance_ppm"]

molecule_mass_variants = pd.read_parquet(CG_INTERIM / "molecule_mass_variants.parquet")
library_fp = artifact_fingerprint("molecule_mass_variants", CG_INTERIM)
mass_index, _, _ = cached_mass_index(
    CG_INTERIM, build_fn=lambda: MassIndex.from_dataframe(molecule_mass_variants, mass_col="exact_mass", key_col="mass_variant_id"),
    library_fingerprint=library_fp, mass_col="exact_mass", key_col="mass_variant_id",
)
variant_to_connectivity = molecule_mass_variants.set_index("mass_variant_id")["connectivity_key"]
gen = CandidateGenerator(mass_index)

reference_index = build_reference_index(train_meta, connectivity_col="connectivity_key", id_col="train_spectrum_id")
spectrum_id_to_ingest_lib = train_meta.set_index("train_spectrum_id")["ingest_lib"].to_dict()
spectrum_id_to_adduct = train_meta.set_index("train_spectrum_id")["adduct"].to_dict()
spectrum_id_to_ce_mean = train_meta.set_index("train_spectrum_id")["ce_mean"].to_dict()

host_pool_raw = gen.generate_dataframe(
    host_holdout_spectra, tolerance_ppm=SELECTED_PPM,
    query_id_col="query_id", mass_col="neutral_mass", true_key_col="true_connectivity_key",
)
host_pool = dedupe_pool_to_connectivity(host_pool_raw, variant_to_connectivity)
print(f"host_pool: {len(host_pool):,} (query, candidate) pairs over {host_pool['query_id'].nunique():,} / {len(host_holdout_spectra):,} host queries")
checks.append(CheckResult("host holdout + candidate pool reloaded", len(host_pool) > 0, f"{len(host_pool):,} pairs"))

[CACHE HIT] mass_index <- C:\Users\myben\OneDrive\Documents\Enveda\data\interim\candidate_generation\mass_index.npz (library_fingerprint matches)


host_pool: 126,573 (query, candidate) pairs over 1,184 / 1,184 host queries


## 2. Section 1.4-1.5 — deterministic, filter-BEFORE-cap reference selection (the actual fix)

ONE pass over each candidate's FULL (uncapped) reference list per (query, candidate) row --
classifies every eligible reference into all 3 protocols simultaneously (not 3 independent
re-scans of the same list, which is what made the original unfixed version take 40+ minutes),
ranks by `(same_adduct desc, collision_energy_difference asc, reference_spectrum_id asc)`,
and only THEN takes the top `MAX_REFS_PER_CANDIDATE`. Near-duplicate flags (section 3 below)
only ever apply to a query's OWN true-connectivity references (see 09b/v1's own note: a
different candidate's reference spectra physically cannot be near-duplicates of the query --
they're different molecules).

In [3]:
def offset_of(spectrum_id):
    return int(spectrum_id.split("_")[1])


def select_protocol_references(candidate_key, query_id, query_adduct, query_ce_mean, near_dup_lookup, limit=MAX_REFS_PER_CANDIDATE):
    """Single pass over the FULL reference list -- returns (selected_by_protocol, n_eligible_by_protocol)
    where selected_by_protocol = {"standard": [...], "cross_library": [...], "duplicate_aware": [...]}
    each already ranked-and-capped, and n_eligible_by_protocol is the PRE-cap eligible count per
    protocol (needed for the section 3 coverage audit -- computed as a free byproduct here, not a
    second pass)."""
    scored = {"standard": [], "cross_library": [], "duplicate_aware": []}
    for rid in reference_index.get(candidate_key, ()):
        if rid == query_id:
            continue
        adduct = spectrum_id_to_adduct.get(rid)
        ce = spectrum_id_to_ce_mean.get(rid)
        same_adduct = adduct == query_adduct
        ce_diff = abs(ce - query_ce_mean) if (pd.notna(ce) and pd.notna(query_ce_mean)) else float("inf")
        sort_key = (0 if same_adduct else 1, ce_diff, rid)

        scored["standard"].append((sort_key, rid))
        if spectrum_id_to_ingest_lib.get(rid) == HOST_INGEST_LIB:
            continue
        scored["cross_library"].append((sort_key, rid))
        if (query_id, rid) in near_dup_lookup:
            continue
        scored["duplicate_aware"].append((sort_key, rid))

    n_eligible = {p: len(v) for p, v in scored.items()}
    selected = {}
    for protocol, items in scored.items():
        items.sort(key=lambda t: t[0])
        selected[protocol] = [rid for _, rid in items[:limit]]
    return selected, n_eligible


checks.append(CheckResult("filter-before-cap reference selection implemented", True,
                           "single pass over the full uncapped reference list per row; cap applied only after ranking"))

## 3. Pass 1 — metadata-only selection for ALL rows (no peaks needed yet)

`duplicate_aware` is a placeholder (=`cross_library`) in this pass -- corrected for TRUE-candidate rows only in section 5, after near-duplicate flags are known (decoy candidates can never be near-duplicates of the query -- they're a different molecule's spectra by definition -- so their `duplicate_aware` selection is already final here).

In [4]:
t0 = time.time()
row_selections = {}   # (query_id, candidate_key) -> {"standard": [...], "cross_library": [...], "duplicate_aware": [...]}
row_eligible = {}      # (query_id, candidate_key) -> {"standard": n, "cross_library": n, "duplicate_aware": n}
query_meta = host_holdout_spectra.set_index("query_id")[["adduct", "ce_mean", "true_connectivity_key"]]

for row in host_pool.itertuples(index=False):
    qmeta = query_meta.loc[row.query_id]
    selected, n_elig = select_protocol_references(
        row.candidate_connectivity_key, row.query_id, qmeta["adduct"], qmeta["ce_mean"], near_dup_lookup={},
    )
    key = (row.query_id, row.candidate_connectivity_key)
    row_selections[key] = selected
    row_eligible[key] = n_elig

print(f"pass 1 (metadata-only, all {len(host_pool):,} rows): {time.time() - t0:.1f}s")
checks.append(CheckResult("pass-1 metadata-only selection complete", len(row_selections) == len(host_pool), ""))

pass 1 (metadata-only, all 126,573 rows): 54.6s


## 4. Section 3 — broader near-duplicate candidate pool for TRUE rows (metadata-only, still no peaks)

In [5]:
NEAR_DUP_POOL_SIZE = 50
true_rows = host_pool[host_pool["is_true_candidate"]]
true_row_near_dup_pool = {}
for row in true_rows.itertuples(index=False):
    qmeta = query_meta.loc[row.query_id]
    selected_broad, _ = select_protocol_references(
        row.candidate_connectivity_key, row.query_id, qmeta["adduct"], qmeta["ce_mean"], near_dup_lookup={},
        limit=NEAR_DUP_POOL_SIZE,
    )
    true_row_near_dup_pool[row.query_id] = selected_broad["standard"]  # source-unfiltered: a technical replicate can be same- or cross-source

print(f"near-dup candidate pool built for {len(true_row_near_dup_pool):,} true-candidate rows (up to {NEAR_DUP_POOL_SIZE} refs each, adduct/CE-ranked)")
checks.append(CheckResult("near-dup candidate pool determined (ranked, not arbitrary file order)", True, f"{len(true_row_near_dup_pool)} rows"))

near-dup candidate pool built for 1,184 true-candidate rows (up to 50 refs each, adduct/CE-ranked)


## 5. Load peaks (ONE combined call: host queries + every reference selected/pooled above)

In [6]:
needed_ref_ids = set()
for selected in row_selections.values():
    for ids in selected.values():
        needed_ref_ids.update(ids)
for pool in true_row_near_dup_pool.values():
    needed_ref_ids.update(pool)

host_query_offsets = set(host_holdout_spectra["query_id"].apply(offset_of))
host_ref_offsets = set(offset_of(i) for i in needed_ref_ids)
host_all_offsets = host_query_offsets | host_ref_offsets
print(f"host: {len(host_query_offsets):,} query offsets + {len(host_ref_offsets):,} reference offsets = {len(host_all_offsets):,} total")

t0 = time.time()
raw = load_rows_by_offset(paths.train, ["ms2_mzs", "ms2_normalized_intensities", "precursor_mz"], list(host_all_offsets))
raw = raw.rename(columns={"_row_offset": "row_offset"})
host_peaks_by_offset = {}
for r in raw.itertuples(index=False):
    mzs, ints = remove_invalid_peaks(r.ms2_mzs, r.ms2_normalized_intensities)
    mzs, ints = truncate_top_peaks(mzs, ints, max_peaks=MAX_PEAKS_PER_SPECTRUM)
    host_peaks_by_offset[r.row_offset] = {"mzs": mzs, "intensities": ints, "precursor_mz": r.precursor_mz}
print(f"loaded {len(host_peaks_by_offset):,} host-side peak rows in {time.time() - t0:.1f}s")
checks.append(CheckResult("host peak arrays loaded (ranked selection, not file-order cap)", len(host_peaks_by_offset) > 0, f"{len(host_peaks_by_offset):,} spectra"))

host: 1,184 query offsets + 167,604 reference offsets = 167,774 total


loaded 167,774 host-side peak rows in 21.8s


## 6. Section 4 — near-duplicate audit (broad, ranked candidate pool) + source breakdown

In [7]:
near_dup_rows = []
for query_row in host_holdout_spectra.itertuples(index=False):
    pool = true_row_near_dup_pool.get(query_row.query_id, [])
    qo = offset_of(query_row.query_id)
    if qo not in host_peaks_by_offset or not pool:
        continue
    q_peaks = host_peaks_by_offset[qo]
    ref_id_to_peaks = {rid: host_peaks_by_offset[offset_of(rid)] for rid in pool if offset_of(rid) in host_peaks_by_offset}
    flagged = find_near_duplicate_references(q_peaks, ref_id_to_peaks, cosine_threshold=DEDUP_COSINE_THRESHOLD, precursor_diff_da=DEDUP_PRECURSOR_DIFF_DA)
    for rid in ref_id_to_peaks:
        near_dup_rows.append({
            "query_id": query_row.query_id, "reference_spectrum_id": rid,
            "reference_ingest_lib": spectrum_id_to_ingest_lib.get(rid),
            "is_same_source": spectrum_id_to_ingest_lib.get(rid) == HOST_INGEST_LIB,
            "is_near_duplicate_reference": rid in flagged,
        })
near_duplicate_audit = pd.DataFrame(near_dup_rows)
near_dup_lookup = set(
    near_duplicate_audit.loc[near_duplicate_audit["is_near_duplicate_reference"], ["query_id", "reference_spectrum_id"]]
    .itertuples(index=False, name=None)
)

n_flagged = int(near_duplicate_audit["is_near_duplicate_reference"].sum())
n_flagged_same_source = int((near_duplicate_audit["is_near_duplicate_reference"] & near_duplicate_audit["is_same_source"]).sum())
n_flagged_cross_source = n_flagged - n_flagged_same_source
n_queries_with_cross_source_dup = near_duplicate_audit.loc[
    near_duplicate_audit["is_near_duplicate_reference"] & ~near_duplicate_audit["is_same_source"], "query_id"
].nunique()

print(f"near-duplicate audit (broad, ranked pool): {len(near_duplicate_audit):,} (query, reference) pairs checked, {n_flagged:,} flagged")
print(f"  near_duplicate AND same_source:  {n_flagged_same_source:,}")
print(f"  near_duplicate AND cross_source: {n_flagged_cross_source:,}")
print(f"  host queries with >=1 CROSS-SOURCE near duplicate: {n_queries_with_cross_source_dup:,} / {len(all_host_query_ids):,} ({n_queries_with_cross_source_dup / len(all_host_query_ids):.2%})")

save_artifact(near_duplicate_audit, "host_near_duplicate_reference_audit", HOST_PROCESSED,
              description="Section 4 (v2): near-duplicate audit with same/cross-source breakdown, ranked (not file-order) candidate pool")
checks.append(CheckResult("near-duplicate audit with source breakdown computed", True, f"{n_flagged_same_source} same-source, {n_flagged_cross_source} cross-source"))

near-duplicate audit (broad, ranked pool): 56,614 (query, reference) pairs checked, 4,102 flagged
  near_duplicate AND same_source:  235
  near_duplicate AND cross_source: 3,867
  host queries with >=1 CROSS-SOURCE near duplicate: 517 / 1,184 (43.67%)


## 7. Section 5 — recompute `duplicate_aware` for TRUE rows only (decoys already final)

In [8]:
for row in true_rows.itertuples(index=False):
    qmeta = query_meta.loc[row.query_id]
    key = (row.query_id, row.candidate_connectivity_key)
    selected, n_elig = select_protocol_references(
        row.candidate_connectivity_key, row.query_id, qmeta["adduct"], qmeta["ce_mean"], near_dup_lookup=near_dup_lookup,
    )
    row_selections[key]["duplicate_aware"] = selected["duplicate_aware"]
    row_eligible[key]["duplicate_aware"] = n_elig["duplicate_aware"]

checks.append(CheckResult("duplicate_aware selection corrected for true-candidate rows", True, f"{len(true_rows)} rows"))

## 8. Section 2/3 — reference-coverage audit (Part 2's old-cap-bug quantification already ran standalone before this notebook was written; this section reports the CORRECTED coverage)

In [9]:
coverage_rows = []
for key, n_elig in row_eligible.items():
    query_id, candidate_key = key
    is_true = candidate_key == query_meta.loc[query_id, "true_connectivity_key"]
    for protocol, n in n_elig.items():
        coverage_rows.append({"query_id": query_id, "candidate_connectivity_key": candidate_key, "is_true_candidate": is_true,
                               "protocol": protocol, "n_eligible": n})
coverage_df = pd.DataFrame(coverage_rows)

coverage_summary = []
for protocol in ["standard", "cross_library", "duplicate_aware"]:
    sub = coverage_df[(coverage_df["protocol"] == protocol) & coverage_df["is_true_candidate"]]
    coverage_summary.append({
        "protocol": protocol,
        "frac_true_with_ge1_ref": (sub["n_eligible"] >= 1).mean(),
        "frac_true_with_ge3_refs": (sub["n_eligible"] >= 3).mean(),
        "frac_true_with_ge5_refs": (sub["n_eligible"] >= MAX_REFS_PER_CANDIDATE).mean(),
        "n_true_with_zero_refs": int((sub["n_eligible"] == 0).sum()),
    })
reference_coverage = pd.DataFrame(coverage_summary)
display(reference_coverage)

zero_ref_fraction = reference_coverage.set_index("protocol").loc["duplicate_aware", "n_true_with_zero_refs"] / len(all_host_query_ids)
print(f"\nduplicate_aware: {reference_coverage.set_index('protocol').loc['duplicate_aware', 'n_true_with_zero_refs']} / {len(all_host_query_ids)} "
      f"true candidates ({zero_ref_fraction:.2%}) have ZERO eligible direct references -- REFERENCE-UNAVAILABLE SUBGROUP")
if zero_ref_fraction > 0.05:
    print("*** >5% of host queries have no eligible true reference under the realistic protocol -- report separately, not as one homogeneous Class-1 set ***")
else:
    print("(below the 5% threshold -- host set can be treated as one homogeneous Class-1 problem for now)")

save_artifact(coverage_df, "host_reference_coverage", HOST_PROCESSED, description="Per-row, per-protocol eligible-reference counts (pre-cap)")
checks.append(CheckResult("reference coverage audit computed", True, f"duplicate_aware zero-ref true candidates: {reference_coverage.set_index('protocol').loc['duplicate_aware', 'n_true_with_zero_refs']}"))

,protocol,frac_true_with_ge1_ref,frac_true_with_ge3_refs,frac_true_with_ge5_refs,n_true_with_zero_refs
0,standard,1.0,1.000000,0.997466,0
1,cross_library,1.0,0.993243,0.989865,0
2,duplicate_aware,1.0,0.993243,0.989865,0



duplicate_aware: 0 / 1184 true candidates (0.00%) have ZERO eligible direct references -- REFERENCE-UNAVAILABLE SUBGROUP
(below the 5% threshold -- host set can be treated as one homogeneous Class-1 problem for now)


## 9. Build pair features for all 3 protocols from the CORRECTED selections (spectral scores computed once per unique reference used)

In [10]:
def offset_ok(rid):
    return offset_of(rid) in host_peaks_by_offset


t0 = time.time()
records = {"standard": [], "cross_library": [], "duplicate_aware": []}
host_fold_by_query = host_holdout_spectra.set_index("query_id")["fold"].to_dict()

for row in host_pool.itertuples(index=False):
    q_offset = offset_of(row.query_id)
    if q_offset not in host_peaks_by_offset:
        continue
    q_peaks = host_peaks_by_offset[q_offset]
    key = (row.query_id, row.candidate_connectivity_key)
    selected = row_selections[key]
    for protocol, ref_ids in selected.items():
        ref_peaks_list = [host_peaks_by_offset[offset_of(rid)] for rid in ref_ids if offset_ok(rid)]
        scores = compute_pair_spectral_scores(q_peaks, ref_peaks_list, bin_width=BIN_WIDTH_DA, peak_tol_da=PEAK_TOL_DA)
        record = {
            "query_id": row.query_id, "candidate_connectivity_key": row.candidate_connectivity_key,
            "abs_mass_error_ppm": row.abs_mass_error_ppm, "is_true_candidate": bool(row.is_true_candidate),
            "fold": host_fold_by_query[row.query_id],
        }
        record.update(scores)
        records[protocol].append(record)

host_pairs_standard = pd.DataFrame.from_records(records["standard"])
host_pairs_cross_library = pd.DataFrame.from_records(records["cross_library"])
host_pairs_duplicate_aware = pd.DataFrame.from_records(records["duplicate_aware"])
print(f"built 3 corrected protocol pair-feature tables in {time.time() - t0:.1f}s: "
      f"standard={len(host_pairs_standard):,}, cross_library={len(host_pairs_cross_library):,}, duplicate_aware={len(host_pairs_duplicate_aware):,} rows")
for name, df in [("host_pair_features_standard", host_pairs_standard), ("host_pair_features_cross_library", host_pairs_cross_library),
                  ("host_pair_features_duplicate_aware", host_pairs_duplicate_aware)]:
    save_artifact(df, name, HOST_INTERIM, description=f"Host holdout pair features (v2, corrected filter-before-cap selection), {name.split('_', 3)[-1]} protocol")
checks.append(CheckResult("3 reference protocols rebuilt with corrected selection", all(len(d) > 0 for d in [host_pairs_standard, host_pairs_cross_library, host_pairs_duplicate_aware]), ""))

built 3 corrected protocol pair-feature tables in 412.4s: standard=126,573, cross_library=126,573, duplicate_aware=126,573 rows


## 10. Reproduce notebook 09's exact fold models (unchanged -- audit subject, not a new model)

In [11]:
pair_features_dev_known = pd.read_parquet(RANK_INTERIM / "pair_features_dev_known_spectrum.parquet")
dev_ranking_queries = pd.read_parquet(RANK_INTERIM / "dev_ranking_queries.parquet")
train_table = pair_features_dev_known.merge(dev_ranking_queries[["query_id", "fold", "true_connectivity_key"]], on="query_id", how="left")
assert_no_prohibited_features(train_table[MODEL_FEATURES], SPECTRAL_RANKING_FEATURE_POLICY)

fold_models = {}
for held_out_fold in sorted(train_table["fold"].unique()):
    tr = train_table[train_table["fold"] != held_out_fold].sort_values("query_id")
    tr_group = tr.groupby("query_id", sort=True).size().to_numpy()
    model = lgb.LGBMRanker(**LGBM_PARAMS)
    model.fit(tr[MODEL_FEATURES], tr["is_true_candidate"].astype(int), group=tr_group)
    fold_models[held_out_fold] = model

mode_a_mrr25_registered = load_promoted_baseline("notebook_09_lambdamart", BASELINE_REGISTRY_PATH, regime="mode_a")["mrr_at_25"]
print(f"reproduced {len(fold_models)} fold models (registered notebook 09 Mode A MRR@25 for reference: {mode_a_mrr25_registered:.4f})")
checks.append(CheckResult("fold models reproduced", len(fold_models) == 5, ""))

reproduced 5 fold models (registered notebook 09 Mode A MRR@25 for reference: 0.7531)


## 11. Score each corrected protocol: mass-only, peak-overlap, LambdaMART

In [12]:
def score_protocol(pair_df):
    lgbm_scores = pd.Series(np.nan, index=pair_df.index)
    for fold_value, model in fold_models.items():
        mask = pair_df["fold"] == fold_value
        if mask.any():
            lgbm_scores.loc[mask] = model.predict(pair_df.loc[mask, MODEL_FEATURES])
    assert lgbm_scores.notna().all(), "every host pair row must be scored by its OWN fold's held-out model"

    results = {}
    for method_name, series in [
        ("mass_only", -pair_df["abs_mass_error_ppm"]),
        ("peak_overlap_max", pair_df["peak_overlap_frac_max"]),
        ("lambdamart", lgbm_scores),
    ]:
        ranked = rank_candidates(pair_df.assign(score=series), score_col="score", ascending=False)
        metrics = ranking_metrics(ranked, all_host_query_ids, k_values=(1, 5, 10, 25))
        ranks = true_candidate_rank(ranked).reindex(all_host_query_ids)
        rr = per_query_reciprocal_rank(ranks, k=25)
        results[method_name] = {"metrics": metrics, "per_query_rr": rr}
    return results


protocol_results = {}
for protocol_name, pair_df in [("standard", host_pairs_standard), ("cross_library", host_pairs_cross_library), ("duplicate_aware", host_pairs_duplicate_aware)]:
    protocol_results[protocol_name] = score_protocol(pair_df)

host_summary_rows = []
for protocol_name, methods in protocol_results.items():
    for method_name, r in methods.items():
        e2e = r["metrics"]["end_to_end"]
        host_summary_rows.append({
            "protocol": protocol_name, "method": method_name, "mrr_at_25": e2e["mrr_at_25"],
            "hit_at_1": e2e["hit_at_1"], "hit_at_5": e2e["hit_at_5"], "hit_at_25": e2e["hit_at_25"],
            "candidate_recall": e2e["candidate_recall"], "n_queries": e2e["n_queries"],
        })
host_mode_a_metrics = pd.DataFrame(host_summary_rows)
display(host_mode_a_metrics)
checks.append(CheckResult("host Mode-A scored (corrected selection) under all 3 protocols x 3 methods", len(host_mode_a_metrics) == 9, ""))

,protocol,method,mrr_at_25,hit_at_1,hit_at_5,hit_at_25,candidate_recall,n_queries
0,standard,mass_only,0.399144,0.214527,0.647804,0.915541,1.0,1184
1,standard,peak_overlap_max,0.668121,0.592905,0.765203,0.872466,1.0,1184
2,standard,lambdamart,0.766280,0.674831,0.878378,0.983953,1.0,1184
3,cross_library,mass_only,0.399144,0.214527,0.647804,0.915541,1.0,1184
4,cross_library,peak_overlap_max,0.574250,0.472973,0.705236,0.851351,1.0,1184
5,cross_library,lambdamart,0.712914,0.596284,0.858953,0.981419,1.0,1184
6,duplicate_aware,mass_only,0.399144,0.214527,0.647804,0.915541,1.0,1184
7,duplicate_aware,peak_overlap_max,0.564917,0.464527,0.692568,0.844595,1.0,1184
8,duplicate_aware,lambdamart,0.723274,0.612331,0.858953,0.983108,1.0,1184


## 12. Paired bootstrap CIs (spectrum-level -- cluster/connectivity-level bootstrap explicitly deferred, see closing note)

In [13]:
primary = protocol_results["duplicate_aware"]
bootstrap_rows = []
for a_name, b_name in [("lambdamart", "peak_overlap_max"), ("lambdamart", "mass_only"), ("peak_overlap_max", "mass_only")]:
    row = paired_bootstrap_report(
        "reciprocal_rank@25", a_name, primary[a_name]["per_query_rr"], b_name, primary[b_name]["per_query_rr"],
        n_bootstrap=N_BOOTSTRAP, seed=SEED,
    )
    row["protocol"] = "duplicate_aware"
    bootstrap_rows.append(row)
paired_bootstrap_comparisons = pd.DataFrame(bootstrap_rows)
display(paired_bootstrap_comparisons[["protocol", "method_a", "method_b", "mean_a", "mean_b", "mean_diff", "ci_low", "ci_high", "significant_at_95"]])

lgbm_vs_peak_overlap = paired_bootstrap_comparisons[
    (paired_bootstrap_comparisons["method_a"] == "lambdamart") & (paired_bootstrap_comparisons["method_b"] == "peak_overlap_max")
].iloc[0]
print(f"\nLambdaMART vs peak_overlap_max (CORRECTED selection): diff={lgbm_vs_peak_overlap['mean_diff']:+.4f} "
      f"[{lgbm_vs_peak_overlap['ci_low']:+.4f}, {lgbm_vs_peak_overlap['ci_high']:+.4f}] "
      f"-- {'SIGNIFICANT' if lgbm_vs_peak_overlap['significant_at_95'] else 'NOT significant'} (spectrum-level CI, NOTE caveat above)")
checks.append(CheckResult("paired bootstrap CIs computed (corrected selection)", len(paired_bootstrap_comparisons) == 3, ""))

,protocol,method_a,method_b,mean_a,mean_b,mean_diff,ci_low,ci_high,significant_at_95
0,duplicate_aware,lambdamart,peak_overlap_max,0.723274,0.564917,0.158357,0.138383,0.179002,True
1,duplicate_aware,lambdamart,mass_only,0.723274,0.399144,0.324130,0.300120,0.346346,True
2,duplicate_aware,peak_overlap_max,mass_only,0.564917,0.399144,0.165773,0.138619,0.191665,True



LambdaMART vs peak_overlap_max (CORRECTED selection): diff=+0.1584 [+0.1384, +0.1790] -- SIGNIFICANT (spectrum-level CI, NOTE caveat above)


## 13. Persist + register corrected baseline + compare against v1's (buggy-selection) result

In [14]:
save_artifact(host_mode_a_metrics, "host_mode_a_metrics", HOST_PROCESSED,
              description="Host holdout Mode-A MRR@25/Hit@K (v2, corrected filter-before-cap selection)")
save_artifact(paired_bootstrap_comparisons, "paired_bootstrap_comparisons", HOST_PROCESSED,
              description="Paired bootstrap CIs, v2 corrected selection, duplicate_aware protocol")

host_lgbm_mrr25 = host_mode_a_metrics.query("protocol == 'duplicate_aware' and method == 'lambdamart'")["mrr_at_25"].iloc[0]
optimism_gap = mode_a_mrr25_registered - host_lgbm_mrr25

v1_metrics = load_baseline("notebook_10_host_holdout_mode_a", BASELINE_REGISTRY_PATH)
v1_mrr25 = v1_metrics["mode_a_host_duplicate_aware"]["mrr_at_25"] if v1_metrics else None
print(f"v1 (file-order-capped selection) duplicate_aware LambdaMART MRR@25: {v1_mrr25}")
print(f"v2 (corrected, ranked selection)  duplicate_aware LambdaMART MRR@25: {host_lgbm_mrr25:.4f}")
if v1_mrr25 is not None:
    print(f"difference (v2 - v1): {host_lgbm_mrr25 - v1_mrr25:+.4f} -- the reference-cap bug's actual effect on the headline number")

register_baseline("notebook_10v2_host_holdout_mode_a_corrected", {
    "mode_a_host_duplicate_aware": {
        "mrr_at_25": float(host_lgbm_mrr25), "status": "VALID_HOST_BENCHMARK_PROVISIONAL", "use_for_model_selection": True,
        "protocol": "cross_library + near_duplicate_excluded (filter-before-cap, adduct/CE-ranked reference selection)",
        "n_queries": int(len(all_host_query_ids)), "n_connectivities": int(host_holdout_spectra['true_connectivity_key'].nunique()),
        "vs_dev_lambdamart_optimism_gap": float(optimism_gap),
        "v1_mrr25_for_comparison": v1_mrr25, "supersedes": "notebook_10_host_holdout_mode_a",
        "not_yet_frozen_reasons": [
            "train/eval reference-protocol mismatch not fixed (LambdaMART trained on notebook 04's standard-protocol features)",
            "source+structure-held-out training variant not evaluated",
            "spectrum-level bootstrap, not connectivity-cluster bootstrap",
            "no transparent mass-likelihood fusion baseline yet",
        ],
    },
    "protocol_comparison": host_mode_a_metrics.to_dict("records"),
    "reference_coverage": reference_coverage.to_dict("records"),
    "near_duplicate_breakdown": {"same_source": n_flagged_same_source, "cross_source": n_flagged_cross_source,
                                   "n_queries_with_cross_source_dup": int(n_queries_with_cross_source_dup)},
}, BASELINE_REGISTRY_PATH, source_notebook="10v2_host_holdout_mode_a_corrected.ipynb", overwrite=True)

with open(BASELINE_REGISTRY_PATH, encoding="utf-8") as f:
    full_registry = json.load(f)
with open(REPORTS_DIR / "baseline_registry.json", "w", encoding="utf-8") as f:
    json.dump(full_registry, f, indent=2, default=str)
print(f"\nBaseline registered in {BASELINE_REGISTRY_PATH} (status: VALID_HOST_BENCHMARK_PROVISIONAL -- NOT frozen, see not_yet_frozen_reasons)")
checks.append(CheckResult("corrected host holdout results registered, v1 preserved for comparison", True, f"v1={v1_mrr25}, v2={host_lgbm_mrr25:.4f}"))

v1 (file-order-capped selection) duplicate_aware LambdaMART MRR@25: 0.7065481387028807
v2 (corrected, ranked selection)  duplicate_aware LambdaMART MRR@25: 0.7233
difference (v2 - v1): +0.0167 -- the reference-cap bug's actual effect on the headline number

Baseline registered in C:\Users\myben\OneDrive\Documents\Enveda\data\processed\baseline_registry.json (status: VALID_HOST_BENCHMARK_PROVISIONAL -- NOT frozen, see not_yet_frozen_reasons)


## 14. Final validation checklist + summary

In [15]:
all_passed, checks_dict = summarize_checks(checks)
checklist_df = pd.DataFrame([{"Check": name, "Status": "PASS" if ok else "FAIL"} for name, ok in checks_dict.items()])
display(checklist_df)

TOTAL_RUNTIME_S = time.time() - NOTEBOOK_START_TIME
print("=" * 70)
print("HOST HOLDOUT MODE-A -- CORRECTED SELECTION -- SUMMARY")
print("=" * 70)
print(f"\nHost holdout: {len(host_holdout_spectra):,} spectra / {host_holdout_spectra['true_connectivity_key'].nunique():,} connectivities")
print(f"\nProtocol comparison (corrected selection):")
print(host_mode_a_metrics.pivot(index="method", columns="protocol", values="mrr_at_25").to_string())
print(f"\nReference coverage (TRUE candidates):")
print(reference_coverage.to_string(index=False))
print(f"\nNear-duplicates: {n_flagged_same_source} same-source, {n_flagged_cross_source} cross-source "
      f"({n_queries_with_cross_source_dup} / {len(all_host_query_ids)} queries affected by a cross-source duplicate)")
print(f"\nv1 (buggy, file-order-capped) vs v2 (corrected, ranked) LambdaMART MRR@25: "
      f"{v1_mrr25} -> {host_lgbm_mrr25:.4f}" + (f" ({host_lgbm_mrr25 - v1_mrr25:+.4f})" if v1_mrr25 is not None else ""))
print(f"\nLambdaMART vs peak_overlap_max: diff={lgbm_vs_peak_overlap['mean_diff']:+.4f} "
      f"[{lgbm_vs_peak_overlap['ci_low']:+.4f}, {lgbm_vs_peak_overlap['ci_high']:+.4f}] "
      f"-- {'SIGNIFICANT' if lgbm_vs_peak_overlap['significant_at_95'] else 'not significant'}")
print(f"\nOptimism gap (dev {mode_a_mrr25_registered:.4f} -> host {host_lgbm_mrr25:.4f}): {optimism_gap:+.4f}")
print(f"\nHost Mode-A status: PROVISIONAL (not frozen) -- {'ALL CHECKS PASS' if all_passed else 'SOME CHECKS FAILED'}")
print(f"\nRuntime: {TOTAL_RUNTIME_S:.1f}s")
print(f"\nNOT done this notebook (deferred, see spec Parts 5-41):")
print(f"    train/eval protocol-matched retraining (variant A: source-held-out, variant B: source+structure-held-out)")
print(f"    timsTOF mass log-likelihood model + transparent fusion baselines")
print(f"    connectivity-cluster bootstrap (spectrum-level CI above is understated -- spectra within a molecule are correlated)")
print(f"    candidate-density / near-miss analysis, host_decision_log.md")
print(f"    score calibration, molecule-level aggregation, first submission, Notebook 06 external candidates")
print("=" * 70)

,Check,Status
0,host holdout + candidate pool reloaded,PASS
1,filter-before-cap reference selection implemented,PASS
2,pass-1 metadata-only selection complete,PASS
3,"near-dup candidate pool determined (ranked, no...",PASS
4,"host peak arrays loaded (ranked selection, not...",PASS
5,near-duplicate audit with source breakdown com...,PASS
6,duplicate_aware selection corrected for true-c...,PASS
7,reference coverage audit computed,PASS
8,3 reference protocols rebuilt with corrected s...,PASS
9,fold models reproduced,PASS


HOST HOLDOUT MODE-A -- CORRECTED SELECTION -- SUMMARY

Host holdout: 1,184 spectra / 250 connectivities

Protocol comparison (corrected selection):
protocol          cross_library  duplicate_aware  standard
method                                                    
lambdamart             0.712914         0.723274  0.766280
mass_only              0.399144         0.399144  0.399144
peak_overlap_max       0.574250         0.564917  0.668121

Reference coverage (TRUE candidates):
       protocol  frac_true_with_ge1_ref  frac_true_with_ge3_refs  frac_true_with_ge5_refs  n_true_with_zero_refs
       standard                     1.0                 1.000000                 0.997466                      0
  cross_library                     1.0                 0.993243                 0.989865                      0
duplicate_aware                     1.0                 0.993243                 0.989865                      0

Near-duplicates: 235 same-source, 3867 cross-source (517 / 1184 q